In [12]:
# --- Imports ---
from dash import Dash, dcc, html, Input, Output, State, ctx
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64

# Set modern plotting style
plt.style.use('default')
sns.set_palette("husl")

# --- Helper function to convert matplotlib figure to Dash image ---
def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100, facecolor='white')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    return html.Img(src='data:image/png;base64,{}'.format(encoded), style={'max-width': '100%', 'border-radius': '8px'})

# --- Load Data ---
data = pd.read_excel('10_Student_Performance.xlsx')
data['pass_fail'] = np.where(data['G3'] >= 10, 1, 0)

categorical_cols = data.select_dtypes(include=['object']).columns
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True).astype(int)

X = data.drop(['G3', 'pass_fail'], axis=1)
Y = data['pass_fail']

summary = data.describe()

# --- Global variables to store results between pages ---
global_results = {
    'accuracies': {},
    'matrices': {},
    'forecasts': pd.DataFrame(),
    'selected_models': []
}

# --- Custom CSS Styles ---
external_stylesheets = ['https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css']

app = Dash(__name__, suppress_callback_exceptions=True, external_stylesheets=external_stylesheets)

# App layout with enhanced styling
app.layout = html.Div([
    # Header
    html.Div([
        html.H1("Student Performance Analytics Dashboard", 
                style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '10px', 'fontWeight': '700'}),
        html.P("Rihab Touil - 604148", 
               style={'textAlign': 'center', 'color': '#7f8c8d', 'fontSize': '16px', 'marginBottom': '30px'})
    ], style={'backgroundColor': '#f8f9fa', 'padding': '20px', 'borderRadius': '10px', 'marginBottom': '20px'}),
    
    # Navigation Tabs
    dcc.Tabs(id='tabs', value='tab-1', children=[
        dcc.Tab(label='Summary & Model Selection', value='tab-1',
                style={'backgroundColor': '#e9ecef', 'color': '#495057', 'padding': '10px', 'fontWeight': '600'},
                selected_style={'backgroundColor': '#0074D9', 'color': 'white', 'borderBottom': '3px solid #0056b3'}),
        dcc.Tab(label='Accuracy & Metrics', value='tab-2',
                style={'backgroundColor': '#e9ecef', 'color': '#495057', 'padding': '10px', 'fontWeight': '600'},
                selected_style={'backgroundColor': '#28a745', 'color': 'white', 'borderBottom': '3px solid #1e7e34'}),
        dcc.Tab(label='Forecast Table', value='tab-3',
                style={'backgroundColor': '#e9ecef', 'color': '#495057', 'padding': '10px', 'fontWeight': '600'},
                selected_style={'backgroundColor': '#6f42c1', 'color': 'white', 'borderBottom': '3px solid #563d7c'}),
    ], style={'marginBottom': '30px', 'fontSize': '16px'}),
    
    html.Div(id='tabs-content', style={'minHeight': '500px'})
], style={'fontFamily': 'Arial, sans-serif', 'padding': '20px', 'backgroundColor': '#ffffff'})

# --- Page 1: Summary & Model Selection ---
# --- Page 1: Summary & Model Selection ---
page_1_layout = html.Div([
    html.Div([
        html.H2("Data Summary & Model Selection", 
                style={'color': '#2c3e50', 'borderBottom': '2px solid #0074D9', 'paddingBottom': '10px'}),
        
        # Data Summary Section
        html.Div([
            html.Div([
                html.H3("Dataset Overview", style={'color': '#0074D9', 'marginBottom': '15px'}),
                html.Div([
                    html.Div([
                        html.Div([
                            html.I(className="fas fa-database", style={'fontSize': '24px', 'color': '#0074D9'}),
                            html.H4(f"{data.shape[0]}", style={'margin': '10px 0 5px 0', 'color': '#2c3e50'}),
                            html.P("Total Samples", style={'color': '#7f8c8d', 'margin': '0'})
                        ], style={'textAlign': 'center', 'padding': '15px'}),
                    ], style={'flex': '1', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px', 'margin': '5px'}),
                    
                    html.Div([
                        html.Div([
                            html.I(className="fas fa-chart-bar", style={'fontSize': '24px', 'color': '#28a745'}),
                            html.H4(f"{data.shape[1]}", style={'margin': '10px 0 5px 0', 'color': '#2c3e50'}),
                            html.P("Features", style={'color': '#7f8c8d', 'margin': '0'})
                        ], style={'textAlign': 'center', 'padding': '15px'}),
                    ], style={'flex': '1', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px', 'margin': '5px'}),
                    
                    html.Div([
                        html.Div([
                            html.I(className="fas fa-graduation-cap", style={'fontSize': '24px', 'color': '#ff6b6b'}),
                            html.H4(f"{(Y.mean() * 100):.1f}%", style={'margin': '10px 0 5px 0', 'color': '#2c3e50'}),
                            html.P("Pass Rate", style={'color': '#7f8c8d', 'margin': '0'})
                        ], style={'textAlign': 'center', 'padding': '15px'}),
                    ], style={'flex': '1', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px', 'margin': '5px'}),
                    
                    html.Div([
                        html.Div([
                            html.I(className="fas fa-exclamation-triangle", style={'fontSize': '24px', 'color': '#fd7e14'}),
                            html.H4(f"{((1 - Y.mean()) * 100):.1f}%", style={'margin': '10px 0 5px 0', 'color': '#2c3e50'}),
                            html.P("Fail Rate", style={'color': '#7f8c8d', 'margin': '0'})
                        ], style={'textAlign': 'center', 'padding': '15px'}),
                    ], style={'flex': '1', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px', 'margin': '5px'}),
                ], style={'display': 'flex', 'justifyContent': 'space-between', 'gap': '10px', 'marginBottom': '20px'}),
                
                # Statistical Summary Section
                html.Div([
                    html.H4("Statistical Summary", style={'color': '#0074D9', 'marginBottom': '10px'}),
                    html.P("Key statistics for numerical features:", style={'color': '#666', 'marginBottom': '15px'}),
                    
                    # Summary table - limited to first few features for readability
                    html.Div([
                        html.Table(
                            # Header row with statistics names
                            [html.Thead(
                                html.Tr([html.Th("Statistic", style={
                                    'padding': '12px',
                                    'backgroundColor': '#0074D9',
                                    'color': 'white',
                                    'textAlign': 'left',
                                    'position': 'sticky',
                                    'left': 0,
                                    'zIndex': 10
                                })] + [html.Th(col[:20] + "..." if len(col) > 20 else col, style={
                                    'padding': '12px',
                                    'backgroundColor': '#0074D9',
                                    'color': 'white',
                                    'minWidth': '120px'
                                }) for col in summary.columns[:6]])
                            )] +
                            
                            # Table body with summary statistics
                            [html.Tbody([
                                html.Tr([html.Td(stat, style={
                                    'padding': '10px',
                                    'fontWeight': 'bold',
                                    'backgroundColor': '#f8f9fa',
                                    'position': 'sticky',
                                    'left': 0,
                                    'textAlign': 'left'
                                })] + [html.Td(
                                    f"{summary.loc[stat, col]:.4f}" if isinstance(summary.loc[stat, col], (int, float, np.float64, np.int64)) 
                                    else str(summary.loc[stat, col]),
                                    style={
                                        'padding': '10px',
                                        'borderBottom': '1px solid #e0e0e0',
                                        'textAlign': 'center',
                                        'backgroundColor': 'white'
                                    }
                                ) for col in summary.columns[:6]]) for stat in summary.index
                            ])],
                            style={
                                'width': '100%',
                                'borderCollapse': 'collapse',
                                'fontSize': '13px',
                                'fontFamily': 'Arial, sans-serif'
                            }
                        )
                    ], style={
                        'overflowX': 'auto',
                        'border': '1px solid #e0e0e0',
                        'borderRadius': '8px',
                        'maxHeight': '350px',
                        'overflowY': 'auto',
                        'marginTop': '10px',
                        'backgroundColor': 'white'
                    }),
                    
                    html.P(f"Showing first 6 of {len(summary.columns)} features. Scroll horizontally to see more.", 
                           style={'color': '#888', 'fontSize': '12px', 'marginTop': '10px', 'fontStyle': 'italic'})
                ], style={'backgroundColor': '#eef7ff', 'padding': '20px', 'borderRadius': '8px'}),
                
            ], style={'backgroundColor': 'white', 'padding': '25px', 'borderRadius': '10px', 'boxShadow': '0 2px 10px rgba(0,0,0,0.1)', 'marginBottom': '25px'}),
        ]),
        
        # Model Selection Section
        html.Div([
            html.H3("Model Selection & Configuration", style={'color': '#28a745', 'marginBottom': '20px'}),
            
            html.Div([
                html.Label("Select Machine Learning Models:", 
                          style={'fontWeight': '600', 'color': '#495057', 'marginBottom': '10px', 'display': 'block'}),
                dcc.Dropdown(
                    id='model_dropdown',
                    options=[
                        {'label': 'Logistic Regression', 'value': 'lg'},
                        {'label': 'XGBoost', 'value': 'xgb'},
                        {'label': 'LightGBM', 'value': 'gbm'}
                    ],
                    value=[],                
                    multi=True,              
                    placeholder="Choose one or more models...",
                    style={'borderRadius': '8px', 'border': '2px solid #e9ecef'}
                ),
            ], style={'marginBottom': '25px'}),
            
            html.Div(id='hyperparameter_inputs', style={'marginTop': '20px'}),
            
            html.Div([
                html.Button(
                    [
                        html.I(className="fas fa-play-circle", style={'marginRight': '8px'}),
                        " Train Selected Models"
                    ], 
                    id='train_all', 
                    n_clicks=0,
                    style={
                        'marginTop': '20px', 
                        'padding': '12px 30px', 
                        'backgroundColor': '#28a745', 
                        'color': 'white', 
                        'border': 'none',
                        'borderRadius': '8px',
                        'fontSize': '16px',
                        'fontWeight': '600',
                        'cursor': 'pointer',
                        'transition': 'all 0.3s ease'
                    }
                ),
            ], style={'textAlign': 'center'}),
        ], style={'backgroundColor': 'white', 'padding': '25px', 'borderRadius': '10px', 'boxShadow': '0 2px 10px rgba(0,0,0,0.1)'}),
    ]),
    
    html.Div(id='train_all_output', style={'marginTop': '40px'})
])
# --- Page 2: Accuracy & Metrics ---
page_2_layout = html.Div([
    html.Div([
        html.H2("Model Accuracy & Performance Metrics", 
                style={'color': '#2c3e50', 'borderBottom': '2px solid #28a745', 'paddingBottom': '10px', 'marginBottom': '30px'}),
        html.Div(id='accuracy-metrics-content')
    ])
])

# --- Page 3: Forecast Table ---
page_3_layout = html.Div([
    html.Div([
        html.H2("Forecasting Results Analysis", 
                style={'color': '#2c3e50', 'borderBottom': '2px solid #6f42c1', 'paddingBottom': '10px', 'marginBottom': '30px'}),
        html.Div(id='forecast-output')
    ])
])

# --- Tab Navigation Callback ---
@app.callback(
    Output('tabs-content', 'children'),
    Input('tabs', 'value')
)
def render_content(tab):
    if tab == 'tab-1':
        return page_1_layout
    elif tab == 'tab-2':
        return page_2_layout
    elif tab == 'tab-3':
        return page_3_layout

# --- Display Hyperparameters Dynamically ---
@app.callback(
    Output('hyperparameter_inputs', 'children'),
    Input('model_dropdown', 'value')
)
def display_hyperparameters(selected_models):
    children = []
    if not selected_models:
        return html.Div("Select models to configure hyperparameters...", 
                       style={'color': '#6c757d', 'fontStyle': 'italic', 'textAlign': 'center', 'padding': '20px'})

    # Styled input container
    input_style = {
        'padding': '8px 12px',
        'border': '2px solid #e9ecef',
        'borderRadius': '6px',
        'margin': '5px 10px 5px 5px',
        'width': '120px'
    }
    
    label_style = {
        'fontWeight': '600',
        'color': '#495057',
        'marginRight': '5px',
        'display': 'inline-block',
        'width': '120px'
    }

    if 'lg' in selected_models:
        children.append(html.Div([
            html.H4("Logistic Regression Settings", 
                   style={'color': '#0074D9', 'marginBottom': '15px', 'paddingBottom': '5px', 'borderBottom': '1px solid #dee2e6'}),
            html.Div([
                html.Label("C (Regularization):", style=label_style),
                dcc.Input(id='lg_C', type='number', value=1.0, min=0.001, step=0.1, style=input_style),
                
                html.Label("Penalty:"),
                    dcc.Dropdown(
                    id='lg_penalty',
                    options=[
                        {"label": "l1", "value": "l1"},
                        {"label": "l2", "value": "l2"},
                        {"label": "elasticnet", "value": "elasticnet"},
                        {"label": "none", "value": "none"},
                    ],
                    value="l2",
                    clearable=False
                ),
                
                html.Label("Solver:"),
                    dcc.Dropdown(
                    id='lg_solver',
                    options=[
                        {"label": "lbfgs", "value": "lbfgs"},
                        {"label": "liblinear", "value": "liblinear"},
                        {"label": "newton-cg", "value": "newton-cg"},
                        {"label": "sag", "value": "sag"},
                        {"label": "saga", "value": "saga"},
                    ],
                    value="lbfgs",
                    clearable=False
                ),
                
                html.Label("Max Iterations:", style=label_style),
                dcc.Input(id='lg_max_iter', type='number', value=1000, min=100, step=100, style=input_style),
            ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
        ], style={
            'backgroundColor': '#f8f9fa', 
            'padding': '20px', 
            'borderRadius': '8px', 
            'marginBottom': '20px',
            'borderLeft': '4px solid #0074D9'
        }))

    if 'xgb' in selected_models:
        children.append(html.Div([
            html.H4("XGBoost Settings", 
                   style={'color': '#fd7e14', 'marginBottom': '15px', 'paddingBottom': '5px', 'borderBottom': '1px solid #dee2e6'}),
            html.Div([
                html.Label("Max Depth:", style=label_style),
                dcc.Input(id='xgb_max_depth', type='number', value=3, min=1, max=20, style=input_style),
                
                html.Label("Learning Rate:", style=label_style),
                dcc.Input(id='xgb_lr', type='number', value=0.01, min=0.001, max=1, step=0.01, style=input_style),
                
                html.Label("Alpha (L1):", style=label_style),
                dcc.Input(id='xgb_alpha', type='number', value=0, min=0, step=0.1, style=input_style),
                
                html.Label("Subsample:", style=label_style),
                dcc.Input(id='xgb_subsample', type='number', value=0.8, min=0.1, max=1, step=0.1, style=input_style),
            ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
        ], style={
            'backgroundColor': '#f8f9fa', 
            'padding': '20px', 
            'borderRadius': '8px', 
            'marginBottom': '20px',
            'borderLeft': '4px solid #fd7e14'
        }))

    if 'gbm' in selected_models:
        children.append(html.Div([
            html.H4("LightGBM Settings", 
                   style={'color': '#6f42c1', 'marginBottom': '15px', 'paddingBottom': '5px', 'borderBottom': '1px solid #dee2e6'}),
            html.Div([
                html.Label("Max Depth:", style=label_style),
                dcc.Input(id='gbm_max_depth', type='number', value=5, min=1, max=20, style=input_style),
                
                html.Label("Num Leaves:", style=label_style),
                dcc.Input(id='gbm_leaves', type='number', value=31, min=2, max=256, style=input_style),
                
                html.Label("Min Child Samples:", style=label_style),
                dcc.Input(id='gbm_child', type='number', value=20, min=1, max=100, style=input_style),
                
                html.Label("Colsample:", style=label_style),
                dcc.Input(id='gbm_colsample', type='number', value=0.8, min=0.1, max=1, step=0.1, style=input_style),
            ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
        ], style={
            'backgroundColor': '#f8f9fa', 
            'padding': '20px', 
            'borderRadius': '8px', 
            'marginBottom': '20px',
            'borderLeft': '4px solid #6f42c1'
        }))

    return children

# --- Training Function with 5-fold CV, saving fold 0 predictions ---
def train_cv(model, X, Y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    acc_train_list, acc_test_list, cm_list = [], [], []
    fold0_preds = None
    fold0_true = None

    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]

        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

        model.fit(X_train_scaled, Y_train)
        Y_train_pred = model.predict(X_train_scaled)
        Y_test_pred = model.predict(X_test_scaled)

        acc_train_list.append(accuracy_score(Y_train, Y_train_pred))
        acc_test_list.append(accuracy_score(Y_test, Y_test_pred))
        cm_list.append(confusion_matrix(Y_test, Y_test_pred))

        if fold == 0:
            fold0_preds = Y_test_pred
            fold0_true = Y_test.values

    avg_acc_train = np.mean(acc_train_list)
    avg_acc_test = np.mean(acc_test_list)
    avg_cm = np.mean(cm_list, axis=0).astype(int)
    acc_df = pd.DataFrame({"Dataset": ["Train", "Test"], "Accuracy": [avg_acc_train, avg_acc_test]})

    pred_fold0 = pd.DataFrame({"True": fold0_true, "Prediction": fold0_preds})
    return acc_df, avg_cm, pred_fold0

# --- Train Selected Models (Page 1) ---
@app.callback(
    Output('train_all_output', 'children'),
    Input('train_all', 'n_clicks'),
    State('model_dropdown', 'value'),
    prevent_initial_call=True
)
def train_selected_models(n_clicks, selected_models):
    global global_results
    
    if not selected_models:
        return html.Div([
            html.Div([
                html.I(className="fas fa-exclamation-triangle", style={'color': '#ffc107', 'fontSize': '24px', 'marginBottom': '10px'}),
                html.H4("No Models Selected", style={'color': '#856404', 'marginBottom': '10px'}),
                html.P("Please select at least one model to train.", style={'color': '#856404'})
            ], style={'textAlign': 'center', 'padding': '30px', 'backgroundColor': '#fff3cd', 'borderRadius': '8px', 'border': '1px solid #ffeaa7'})
        ])

    # Show loading state
    loading_div = html.Div([
        html.Div([
            html.I(className="fas fa-spinner fa-spin", style={'fontSize': '32px', 'color': '#0074D9', 'marginBottom': '15px'}),
            html.H4("Training Models...", style={'color': '#0074D9', 'marginBottom': '10px'}),
            html.P("Please wait while we train your selected models with 5-fold cross-validation.", style={'color': '#6c757d'})
        ], style={'textAlign': 'center', 'padding': '30px'})
    ])

    # Simulate some processing time for better UX
    import time
    time.sleep(1)

    results = []
    accuracies = {}
    matrices = {}
    forecasts = pd.DataFrame()

    for model_key in selected_models:
        # --- Logistic Regression ---
        if model_key == 'lg':
            lg_C = ctx.states.get('lg_C.value', 1.0)
            lg_penalty = ctx.states.get('lg_penalty.value', 'l2')
            lg_solver = ctx.states.get('lg_solver.value', 'lbfgs')
            lg_max_iter = ctx.states.get('lg_max_iter.value', 1000)

            model = LogisticRegression(
                C=lg_C, penalty=lg_penalty, solver=lg_solver, max_iter=lg_max_iter
            )
            acc_df, cm, pred_fold0 = train_cv(model, X, Y)
            model_name = "Logistic Regression"

        # --- XGBoost ---
        elif model_key == 'xgb':
            xgb_max_depth = ctx.states.get('xgb_max_depth.value', 3)
            xgb_lr = ctx.states.get('xgb_lr.value', 0.01)
            xgb_alpha = ctx.states.get('xgb_alpha.value', 0)
            xgb_subsample = ctx.states.get('xgb_subsample.value', 0.8)

            model = XGBClassifier(
                max_depth=xgb_max_depth, learning_rate=xgb_lr,
                reg_alpha=xgb_alpha, subsample=xgb_subsample,
                use_label_encoder=False, eval_metric='logloss'
            )
            acc_df, cm, pred_fold0 = train_cv(model, X, Y)
            model_name = "XGBoost"

        # --- LightGBM ---
        elif model_key == 'gbm':
            gbm_max_depth = ctx.states.get('gbm_max_depth.value', 5)
            gbm_leaves = ctx.states.get('gbm_leaves.value', 31)
            gbm_child = ctx.states.get('gbm_child.value', 20)
            gbm_colsample = ctx.states.get('gbm_colsample.value', 0.8)

            model = LGBMClassifier(
                max_depth=gbm_max_depth, num_leaves=gbm_leaves,
                min_child_samples=gbm_child, colsample_bytree=gbm_colsample
            )
            acc_df, cm, pred_fold0 = train_cv(model, X, Y)
            model_name = "LightGBM"

        # Store model results
        accuracies[model_name] = acc_df.loc[acc_df["Dataset"] == "Test", "Accuracy"].values[0]
        matrices[model_name] = cm

        # Collect predictions for fold 0
        if forecasts.empty:
            forecasts = pred_fold0.rename(columns={"Prediction": model_name})
        else:
            forecasts[model_name] = pred_fold0["Prediction"]

    # --- Add true values ---
    forecasts["True"] = pred_fold0["True"]

    # --- Compute error count ---
    pred_cols = [c for c in forecasts.columns if c not in ["True"]]
    forecasts["Error_Count"] = (forecasts[pred_cols] != forecasts["True"].values.reshape(-1, 1)).sum(axis=1)

    # --- Sort by Error_Count ---
    forecasts_sorted = forecasts.sort_values(by="Error_Count", ascending=False)

    # --- Reorder columns (True first, Error_Count last) ---
    ordered_cols = ["True"] + pred_cols + ["Error_Count"]
    forecasts_sorted = forecasts_sorted[ordered_cols]

    # --- Store results globally for other pages ---
    global_results['accuracies'] = accuracies
    global_results['matrices'] = matrices
    global_results['forecasts'] = forecasts_sorted
    global_results['selected_models'] = selected_models

    # --- Training completion message ---
    training_complete = html.Div([
        html.Div([
            html.I(className="fas fa-check-circle", style={'fontSize': '48px', 'color': '#28a745', 'marginBottom': '15px'}),
            html.H3("Training Completed Successfully!", style={'color': '#155724', 'marginBottom': '10px'}),
            html.P(f"Successfully trained {len(selected_models)} model(s) with 5-fold cross-validation", 
                   style={'color': '#155724', 'fontSize': '16px', 'marginBottom': '5px'}),
            html.P("Navigate to other tabs to explore detailed performance metrics and predictions", 
                   style={'color': '#155724', 'fontSize': '16px'}),
            
            html.Div([
                html.Div([
                    html.H5("Trained Models", style={'color': '#495057', 'marginBottom': '10px'}),
                    html.Ul([html.Li(model, style={'color': '#495057'}) for model in accuracies.keys()])
                ], style={'flex': '1', 'textAlign': 'left'}),
                
                html.Div([
                    html.H5("Next Steps", style={'color': '#495057', 'marginBottom': '10px'}),
                    html.Ul([
                        html.Li("View accuracy comparisons", style={'color': '#495057'}),
                        html.Li("Analyze confusion matrices", style={'color': '#495057'}),
                        html.Li("Explore prediction results", style={'color': '#495057'})
                    ])
                ], style={'flex': '1', 'textAlign': 'left'})
            ], style={'display': 'flex', 'justifyContent': 'space-between', 'marginTop': '20px', 'gap': '20px'})
        ], style={'textAlign': 'center', 'padding': '30px', 'backgroundColor': '#d4edda', 'borderRadius': '8px', 'border': '1px solid #c3e6cb'})
    ])

    return training_complete

# --- Update Accuracy Metrics Page ---
@app.callback(
    Output('accuracy-metrics-content', 'children'),
    Input('tabs', 'value')
)
def update_accuracy_metrics(tab):
    if tab != 'tab-2' or not global_results['accuracies']:
        return html.Div([
            html.Div([
                html.I(className="fas fa-chart-bar", style={'fontSize': '48px', 'color': '#6c757d', 'marginBottom': '15px'}),
                html.H3("No Training Data Available", style={'color': '#6c757d', 'marginBottom': '10px'}),
                html.P("Please train models first from the 'Summary & Model Selection' tab to view performance metrics.", 
                       style={'color': '#6c757d'})
            ], style={'textAlign': 'center', 'padding': '50px', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px'})
        ])
    
    accuracies = global_results['accuracies']
    matrices = global_results['matrices']
    
    content = []
    
    # Accuracy Bar Plot
    if accuracies:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Bar plot
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
        bars = ax1.bar(accuracies.keys(), accuracies.values(), color=colors[:len(accuracies)], alpha=0.8)
        ax1.set_title("Model Accuracy Comparison (5-Fold CV)", fontsize=16, fontweight='bold', pad=20)
        ax1.set_ylabel("Accuracy Score", fontweight='bold')
        ax1.set_xlabel("Machine Learning Models", fontweight='bold')
        ax1.set_ylim(0, 1)
        ax1.grid(axis='y', alpha=0.3)
        ax1.tick_params(axis='x', rotation=15)
        
        # Add value labels on bars
        for bar, (model, acc) in zip(bars, accuracies.items()):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{acc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
        
        # Pie chart for accuracy distribution
        if len(accuracies) > 1:
            ax2.pie(accuracies.values(), labels=accuracies.keys(), autopct='%1.1f%%', 
                   colors=colors[:len(accuracies)], startangle=90)
            ax2.set_title("Accuracy Distribution", fontsize=16, fontweight='bold', pad=20)
        
        plt.tight_layout()
        
        content.append(html.Div([
            html.H3("Model Performance Overview", 
                   style={'color': '#495057', 'marginBottom': '20px', 'paddingBottom': '10px', 'borderBottom': '2px solid #dee2e6'}),
            fig_to_img(fig)
        ], style={'marginBottom': '30px'}))
        plt.close(fig)
    
    # Confusion Matrices
    if matrices:
        content.append(html.H3("Confusion Matrices Analysis", 
                             style={'color': '#495057', 'marginBottom': '20px', 'paddingBottom': '10px', 'borderBottom': '2px solid #dee2e6'}))
        
        # Create subplots for confusion matrices
        n_models = len(matrices)
        fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4.5))
        if n_models == 1:
            axes = [axes]
        
        for idx, (name, cm) in enumerate(matrices.items()):
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                       annot_kws={'size': 12, 'weight': 'bold'})
            axes[idx].set_title(f'{name}\n', fontsize=14, fontweight='bold')
            axes[idx].set_xlabel('Predicted Label', fontweight='bold')
            axes[idx].set_ylabel('True Label', fontweight='bold')
            axes[idx].tick_params(axis='both', which='major', labelsize=10)
        
        plt.tight_layout()
        content.append(fig_to_img(fig))
        plt.close(fig)
        
    
    return html.Div(content)

# --- Update Forecast Table Page ---
@app.callback(
    Output('forecast-output', 'children'),
    Input('tabs', 'value')
)
def update_forecast_table(tab):
    if tab != 'tab-3' or global_results['forecasts'].empty:
        return html.Div([
            html.Div([
                html.I(className="fas fa-table", style={'fontSize': '48px', 'color': '#6c757d', 'marginBottom': '15px'}),
                html.H3("No Prediction Data Available", style={'color': '#6c757d', 'marginBottom': '10px'}),
                html.P("Please train models first from the 'Summary & Model Selection' tab to view prediction results.", 
                       style={'color': '#6c757d'})
            ], style={'textAlign': 'center', 'padding': '50px', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px'})
        ])
    
    forecasts_sorted = global_results['forecasts']
    
    # Create styled forecast table
    forecast_table = html.Div([
        html.Div([
            html.H3("Detailed Predictions Analysis", 
                   style={'color': '#495057', 'marginBottom': '15px', 'paddingBottom': '10px', 'borderBottom': '2px solid #dee2e6'}),
            html.P("This table displays predictions from the first cross-validation fold, highlighting samples where models disagree.", 
                  style={'color': '#6c757d', 'fontSize': '16px', 'marginBottom': '20px'}),
        ]),
        
        html.Div([
            html.Table(
                [
                    html.Thead(
                        html.Tr([
                            html.Th(col, style={
                                "padding": "15px", 
                                "border": "2px solid #dee2e6",
                                "backgroundColor": "#6f42c1",
                                "color": "white",
                                "textAlign": "center",
                                "fontWeight": "600",
                                "fontSize": "14px"
                            })
                            for col in forecasts_sorted.columns
                        ])
                    ),
                    html.Tbody([
                        html.Tr([
                            html.Td(
                                forecasts_sorted.iloc[i][col],
                                style={
                                    "padding": "12px",
                                    "border": "1px solid #dee2e6",
                                    "textAlign": "center",
                                    "backgroundColor": "#ffebee" if forecasts_sorted.iloc[i]["Error_Count"] > 0 else "#e8f5e8",
                                    "fontWeight": "bold" if col == "Error_Count" else "normal",
                                    "fontSize": "13px",
                                    "transition": "all 0.2s ease"
                                }
                            )
                            for col in forecasts_sorted.columns
                        ], style={'cursor': 'pointer'})
                        for i in range(min(25, len(forecasts_sorted)))  # Show first 25 rows
                    ])
                ],
                style={
                    "borderCollapse": "collapse",
                    "marginTop": "20px",
                    "width": "100%",
                    "boxShadow": "0 4px 6px rgba(0,0,0,0.1)",
                    "borderRadius": "8px",
                    "overflow": "hidden"
                }
                )
        ], style={'marginBottom': '30px'}),
        
        # Summary Statistics
        html.Div([
            html.H4("Prediction Summary", style={'color': '#495057', 'marginBottom': '15px'}),
            html.Div([
                html.Div([
                    html.Div([
                        html.I(className="fas fa-check-circle", style={'fontSize': '20px', 'color': '#28a745', 'marginBottom': '8px'}),
                        html.H5(f"{(forecasts_sorted['Error_Count'] == 0).sum()}", style={'margin': '5px 0', 'color': '#155724', 'fontSize': '24px'}),
                        html.P("Perfect Predictions", style={'color': '#6c757d', 'margin': '0', 'fontSize': '14px'})
                    ], style={'textAlign': 'center', 'padding': '15px'})
                ], style={'flex': '1', 'backgroundColor': '#d4edda', 'borderRadius': '8px', 'margin': '5px', 'border': '1px solid #c3e6cb'}),
                
                html.Div([
                    html.Div([
                        html.I(className="fas fa-exclamation-triangle", style={'fontSize': '20px', 'color': '#ffc107', 'marginBottom': '8px'}),
                        html.H5(f"{(forecasts_sorted['Error_Count'] > 0).sum()}", style={'margin': '5px 0', 'color': '#856404', 'fontSize': '24px'}),
                        html.P("Disagreed Predictions", style={'color': '#6c757d', 'margin': '0', 'fontSize': '14px'})
                    ], style={'textAlign': 'center', 'padding': '15px'})
                ], style={'flex': '1', 'backgroundColor': '#fff3cd', 'borderRadius': '8px', 'margin': '5px', 'border': '1px solid #ffeaa7'}),
                
                html.Div([
                    html.Div([
                        html.I(className="fas fa-times-circle", style={'fontSize': '20px', 'color': '#dc3545', 'marginBottom': '8px'}),
                        html.H5(f"{(forecasts_sorted['Error_Count'] == len(global_results['selected_models'])).sum()}", style={'margin': '5px 0', 'color': '#721c24', 'fontSize': '24px'}),
                        html.P("All Models Wrong", style={'color': '#6c757d', 'margin': '0', 'fontSize': '14px'})
                    ], style={'textAlign': 'center', 'padding': '15px'})
                ], style={'flex': '1', 'backgroundColor': '#f8d7da', 'borderRadius': '8px', 'margin': '5px', 'border': '1px solid #f5c6cb'}),
                
                html.Div([
                    html.Div([
                        html.I(className="fas fa-percentage", style={'fontSize': '20px', 'color': '#0074D9', 'marginBottom': '8px'}),
                        html.H5(f"{((forecasts_sorted['Error_Count'] == 0).sum() / len(forecasts_sorted) * 100):.1f}%", style={'margin': '5px 0', 'color': '#004085', 'fontSize': '24px'}),
                        html.P("Consensus Accuracy", style={'color': '#6c757d', 'margin': '0', 'fontSize': '14px'})
                    ], style={'textAlign': 'center', 'padding': '15px'})
                ], style={'flex': '1', 'backgroundColor': '#cce7ff', 'borderRadius': '8px', 'margin': '5px', 'border': '1px solid #b3d7ff'}),
            ], style={'display': 'flex', 'justifyContent': 'space-between', 'gap': '10px', 'marginBottom': '20px'}),
        ]),
        
        # Model Agreement Analysis
        html.Div([
            html.H4("Model Agreement Analysis", style={'color': '#495057', 'marginBottom': '15px'}),
            html.Div([
                html.Div([
                    html.H5("Model Consensus", style={'color': '#495057', 'marginBottom': '10px'}),
                    html.P("When all models agree on a prediction, it indicates high confidence. Disagreements highlight challenging samples that may require further investigation.", 
                          style={'color': '#6c757d', 'lineHeight': '1.6'})
                ], style={'flex': '2', 'paddingRight': '20px'}),
                
                html.Div([
                    html.H5("Error Count Meaning", style={'color': '#495057', 'marginBottom': '10px'}),
                    html.Ul([
                        html.Li("0: All models predicted correctly", style={'color': '#6c757d'}),
                        html.Li("1-2: Some model disagreement", style={'color': '#6c757d'}),
                        html.Li(f"{len(global_results['selected_models'])}: All models predicted incorrectly", style={'color': '#6c757d'})
                    ])
                ], style={'flex': '1', 'backgroundColor': '#f8f9fa', 'padding': '15px', 'borderRadius': '8px'})
            ], style={'display': 'flex', 'gap': '20px', 'marginBottom': '15px'}),
        ], style={'backgroundColor': '#f8f9fa', 'padding': '20px', 'borderRadius': '8px', 'marginBottom': '20px'}),
        
        html.Div([
            html.P(f"Showing top {min(25, len(forecasts_sorted))} of {len(forecasts_sorted)} predictions sorted by disagreement level", 
                   style={'color': '#6c757d', 'fontSize': '14px', 'marginBottom': '5px'}),
            html.P("Color coding: Green = correct predictions, Red = prediction errors", 
                   style={'color': '#6c757d', 'fontSize': '14px', 'marginBottom': '5px'}),
            html.P("Tip: Focus on high-error-count samples to understand model limitations", 
                   style={'color': '#6c757d', 'fontSize': '14px'})
        ], style={'textAlign': 'center', 'padding': '15px', 'backgroundColor': '#e9ecef', 'borderRadius': '8px'})
    ])
    
    return forecast_table

# --- Run App ---
if __name__ == '__main__':
    app.run(debug=True, dev_tools_ui=True, dev_tools_hot_reload=True)